In [ ]:
import os
import math
import time
import zipfile
import cv2
import torch

# Upload ZIP File
from google.colab import files

print("Upload your ZIP file containing lane images...")
uploaded = files.upload()

zip_file = next(iter(uploaded.keys()))
# Extract ZIP
extract_dir = "/content/test_images"

if os.path.exists(extract_dir):
    import shutil
    shutil.rmtree(extract_dir)

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"ZIP extracted to: {extract_dir}")

# Show extracted structure
print("\nExtracted Files:")
for root, dirs, files_ in os.walk(extract_dir):
    for file in files_:
        print(os.path.join(root, file))

# Load YOLOv5
print("\nLoading YOLOv5...")
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')
# Paths

inputPath = extract_dir
outputPath = "/content/output_images"

os.makedirs(outputPath, exist_ok=True)

# Traffic Parameters
defaultGreen = 20
defaultYellow = 5
defaultMinimum = 10
defaultMaximum = 60

carTime = 2
busTime = 2.5
truckTime = 2.5
motorcycleTime = 1.5
bicycleTime = 1

noOfLanes = 2

allowed_classes = {
    "car",
    "bus",
    "motorcycle",
    "truck",
    "bicycle"
}

# Vehicle Detection
def detectVehicles(image_path):

    vehicle_counts = {vehicle: 0 for vehicle in allowed_classes}

    img = cv2.imread(image_path)

    if img is None:
        print(f"Could not read image: {image_path}")
        return vehicle_counts

    results = model(image_path)

    detections = results.pandas().xyxy[0]

    for _, vehicle in detections.iterrows():

        label = vehicle["name"]

        if label in allowed_classes:

            vehicle_counts[label] += 1

            xmin = int(vehicle["xmin"])
            ymin = int(vehicle["ymin"])
            xmax = int(vehicle["xmax"])
            ymax = int(vehicle["ymax"])

            cv2.rectangle(
                img,
                (xmin, ymin),
                (xmax, ymax),
                (0, 255, 0),
                2
            )

            cv2.putText(
                img,
                label,
                (xmin, ymin - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    output_file = os.path.join(
        outputPath,
        os.path.basename(image_path)
    )

    cv2.imwrite(output_file, img)

    return vehicle_counts

# Green Time Calculation
def calculateGreenTime(vehicle_counts):

    cars = vehicle_counts.get("car", 0)
    motorcycles = vehicle_counts.get("motorcycle", 0)
    bicycles = vehicle_counts.get("bicycle", 0)
    buses = vehicle_counts.get("bus", 0)
    trucks = vehicle_counts.get("truck", 0)

    green_time = math.ceil(
        (
            cars * carTime +
            bicycles * bicycleTime +
            buses * busTime +
            trucks * truckTime +
            motorcycles * motorcycleTime
        )
        / (noOfLanes + 1)
    )

    green_time = max(defaultMinimum, green_time)
    green_time = min(defaultMaximum, green_time)

    return green_time

# Red Time Calculation
def calculateRedTime(current_green, all_green_times):

    return (
        sum(all_green_times)
        - current_green
        + defaultYellow * (len(all_green_times) - 1)
    )

# Process Lanes
def processLanes():

    image_files = []

    for root, dirs, files_ in os.walk(inputPath):

        for file in files_:

            if file.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                image_files.append(
                    os.path.join(root, file)
                )

    image_files.sort()

    if len(image_files) == 0:
        print("\nNo images found in ZIP!")
        return

    print(f"\nFound {len(image_files)} image(s):")

    for img in image_files:
        print(img)

    signal_times = []
    all_green_times = []

    for idx, image_path in enumerate(image_files):

        print("\n" + "="*50)
        print(f"Processing Lane {idx+1}")
        print("="*50)

        vehicle_counts = detectVehicles(image_path)

        print("Detected Vehicles:")
        print(vehicle_counts)

        green_time = calculateGreenTime(vehicle_counts)

        all_green_times.append(green_time)

        red_time = calculateRedTime(
            green_time,
            all_green_times
        )

        signal_times.append(
            (
                red_time,
                green_time,
                defaultYellow
            )
        )

        print(f"Green Time : {green_time} sec")
        print(f"Red Time   : {red_time} sec")
        print(f"Yellow Time: {defaultYellow} sec")

        time.sleep(1)

    print("\n")
    print("="*60)
    print("FINAL SIGNAL TIMINGS")
    print("="*60)

    for lane, times in enumerate(signal_times, start=1):

        print(
            f"Lane {lane}: "
            f"Red={times[0]}s | "
            f"Green={times[1]}s | "
            f"Yellow={times[2]}s"
        )

    signal_file = "/content/signal_times.txt"

    with open(signal_file, "w") as f:

        for lane, times in enumerate(signal_times, start=1):

            f.write(
                f"Lane {lane}: "
                f"Red={times[0]} "
                f"Green={times[1]} "
                f"Yellow={times[2]}\n"
            )

    print("\nResults saved:")
    print(outputPath)
    print(signal_file)

# Run
processLanes()

Upload your ZIP file containing lane images...


Saving tf.zip to tf (2).zip
ZIP extracted to: /content/test_images

Extracted Files:
/content/test_images/tf/1.jpg
/content/test_images/tf/2.jpg

Loading YOLOv5...


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-6-12 Python-3.12.13 torch-2.11.0+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):



Found 2 image(s):
/content/test_images/tf/1.jpg
/content/test_images/tf/2.jpg

Processing Lane 1
Detected Vehicles:
{'car': 27, 'truck': 0, 'bus': 0, 'motorcycle': 0, 'bicycle': 0}
Green Time : 18 sec
Red Time   : 0 sec
Yellow Time: 5 sec

Processing Lane 2


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Detected Vehicles:
{'car': 12, 'truck': 6, 'bus': 0, 'motorcycle': 2, 'bicycle': 0}
Green Time : 14 sec
Red Time   : 23 sec
Yellow Time: 5 sec


FINAL SIGNAL TIMINGS
Lane 1: Red=0s | Green=18s | Yellow=5s
Lane 2: Red=23s | Green=14s | Yellow=5s

Results saved:
/content/output_images
/content/signal_times.txt
